### 踩的坑

ollama获取帮助和打开服务
```bash
ollama --help
ollama serve
```

**打开服务时候报错：**`error: listen tcp 127.0.0.1:11434: bind: address already in use`

**步骤1**：查找占用 11434 端口的进程

```bash
# 查找占用端口的进程（两种方式任选其一）
lsof -i :11434
# 或
ss -tulpn | grep :11434
```

执行后会看到进程名称（通常是 ollama）和 PID，比如：
```plaintext
COMMAND  PID   USER   FD   TYPE             DEVICE SIZE/OFF NODE NAME
ollama   5678  user    3u  IPv4 0x1234567890abcdef      0t0  TCP localhost:11434 (LISTEN)
```


**步骤2**：终止占用端口的进程

使用以下命令终止进程（将 `5678` 替换为实际的 PID）：
```bash
kill -9 5678
```

In [1]:
import json
import ollama
from datetime import datetime
from zoneinfo import ZoneInfo
import requests
from utils import print_html

In [2]:
ollama.list()['models']

[Model(model='glm-4.7-flash:latest', modified_at=datetime.datetime(2026, 2, 28, 17, 21, 33, 127567, tzinfo=TzInfo(28800)), digest='d1a8a26252f18b34301218d22abd2620a65b85ba4b78987842eb157e01321222', size=19019270852, details=ModelDetails(parent_model='', format='gguf', family='glm4moelite', families=['glm4moelite'], parameter_size='29.9B', quantization_level='Q4_K_M')),
 Model(model='deepseek-r1-distill-qwen-32b:latest', modified_at=datetime.datetime(2026, 2, 28, 13, 40, 42, 270811, tzinfo=TzInfo(28800)), digest='d9d62508e77d1c144ddb65cd7dad2fc4aaaaf6772204ee6fef791f39c5a930fb', size=19851337458, details=ModelDetails(parent_model='', format='gguf', family='qwen2', families=['qwen2'], parameter_size='32.8B', quantization_level='Q4_K_M')),
 Model(model='qwen3-coder-30b-q4:latest', modified_at=datetime.datetime(2026, 2, 27, 15, 55, 29, 475586, tzinfo=TzInfo(28800)), digest='1f5fcc4c7888f8206ce65c57f53e287510af8415c819897ba8afe18d6e5b5940', size=17304492943, details=ModelDetails(parent_mode

In [3]:
glm_model = 'glm-4.7-flash'
deepseek_model = 'deepseek-r1-distill-qwen-32b'
qwen_model = 'qwen3-coder-30b-q4'

### 非流式回答

In [4]:
response = ollama.generate(
    model = deepseek_model,
    prompt = '贾宝玉的父亲是谁？',
)
print(response['response'])

贾宝玉的父亲是贾政。贾政是《红楼梦》中的主要人物之一，他是贾家的家长，性格严肃，对贾宝玉要求严格。


### 流式输出

In [5]:
for chunk in ollama.generate(model=qwen_model,prompt='介绍一下 DeepSeek 模型的特点',stream=True):
    print(chunk['response'],end='',flush=True)

DeepSeek 是一款由 DeepSeek 公司开发的大型语言模型，具有以下主要特点：

1. **多语言支持**：
   - DeepSeek 模型支持多种语言，包括中文、英文、德语、法语、西班牙语等，满足全球用户的需求。

2. **强大的推理能力**：
   - DeepSeek 模型在逻辑推理、数学计算、代码生成等方面表现出色，能够处理复杂的任务和问题。

3. **高效的训练和推理**：
   - DeepSeek 模型采用了高效的训练和推理技术，能够在保证模型性能的同时，提升推理速度和效率。

4. **开源社区支持**：
   - DeepSeek 模型的开源社区活跃，用户可以参与模型的改进和优化，共同推动模型的发展。

5. **丰富的应用场景**：
   - DeepSeek 模型可以应用于多种场景，如智能客服、自动写作、代码生成、教育辅导等，满足不同用户的需求。

6. **持续更新和优化**：
   - DeepSeek 团队持续对模型进行更新和优化，不断提升模型的性能和用户体验。

这些特点使得 DeepSeek 模型在多个领域具有广泛的应用前景。

### 多轮对话

In [6]:
# 初始化对话历史（也可以不传，由 SDK 自动维护）
messages = [
    {"role": "user", "content": "Ollama 支持哪些模型格式？"},
]
# 第一轮对话
response = ollama.chat(model=qwen_model, messages=messages)
messages.append({"role": "assistant", "content": response["message"]["content"]})
print("助手：", response["message"]["content"])

# 第二轮对话（基于历史上下文）
messages.append({"role": "user", "content": "这些格式中哪个最适合本地部署？"})
response = ollama.chat(model=deepseek_model, messages=messages, stream=False)
print("助手：", response["message"]["content"])

助手： Ollama 支持多种模型格式，主要包括以下几种：

1. **GGUF**（Gguf General Unified Format）：这是 Ollama 推荐的模型格式，具有高效的推理性能和较小的文件大小。GGUF 是一种轻量级的模型格式，支持多种模型架构（如 LLaMA、Mistral、Phi 等），并且可以包含量化版本（如 Q4_K_M、Q5_K_M 等）。

2. **Hugging Face 格式**：Ollama 也支持从 Hugging Face 下载的模型，通常这些模型以 `.bin` 或 `.safetensors` 文件的形式存在。你可以通过指定 Hugging Face 的模型名称来加载这些模型。

3. **自定义模型格式**：如果你有特定的模型文件，Ollama 也允许你使用自定义的模型格式，但需要确保这些模型与 Ollama 的推理引擎兼容。

4. **ONNX 格式**：Ollama 也支持 ONNX（Open Neural Network Exchange）格式的模型，这是一种开放的模型格式，可以用于跨平台部署。

### 支持的模型架构
Ollama 支持多种常见的模型架构，包括但不限于：

- **LLaMA** 系列（如 LLaMA2、LLaMA3）
- **Mistral** 系列
- **Phi** 系列
- **Qwen** 系列（如 Qwen2、Qwen3）

### 示例：使用 Ollama 加载模型
```bash
# 使用 GGUF 格式的模型
ollama run llama3:8b

# 使用 Hugging Face 模型
ollama run mistral:7b
```

### 总结
Ollama 主要支持 GGUF 和 Hugging Face 格式的模型，同时也支持 ONNX 格式。你可以根据需要选择合适的模型格式进行加载和推理。

如果你有特定的模型格式需求，可以提供更多细节，我可以帮助你进一步分析。
助手： 在 Ollama 支持的模型格式中，GGUF 格式通常是最适合本地部署的选择。以下是几个原因：

1. **高效推理性能**：GGUF 是为高性能推理设计的格式，优化了内存使用和计算速度，能够在本地设备上提供流畅的体验。

2. **较小的文件大小**：通过量化技术（如 Q4_K_

### 函数调用

为什么ollama调用的时候`qwen_model`和`deepseek_model`都失效了？

In [7]:
def get_current_time(timezone):
    """
    获取指定时区的当前时间
    
    Args:
        timezone: IANA 时区名称，如 'Asia/Shanghai', 'UTC', 'America/New_York', 'Europe/Berlin', 'Australia/Sydney'. Default: 'Asia/Shanghai'
    Returns:
        str: 当前时间，格式为 'HH:MM:SS'
    """
    timezone = ZoneInfo(timezone)
    return datetime.now(timezone).strftime("%H:%M:%S")



def get_weather_from_ip():
    """
    自动获取用户当前所在的位置的当前温度、最高与最低温度（摄氏制），并返回给用户。

    Args:
        None
    Returns:
        str: 当前天气信息，格式为 'Current: XX°C, High: XX°C, Low: XX°C'
    """
    # 通过 IP 地址获取地理坐标
    lat, lon = requests.get('https://ipinfo.io/json').json()['loc'].split(',')
    
    # 设置天气 API 调用的参数
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "celsius",
        "timezone": "auto"
    }

    # 获取天气数据
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    # 格式化并返回简洁字符串
    return (
        f"Current: {weather_data['current']['temperature_2m']}°C, "
        f"High: {weather_data['daily']['temperature_2m_max'][0]}°C, "
        f"Low: {weather_data['daily']['temperature_2m_min'][0]}°C"
    )

In [8]:
all_tools = {'get_current_time':get_current_time, 'get_weather_from_ip':get_weather_from_ip}

In [9]:
max_iterations = 3

messages = [{"role": "user", "content": "现在几点了？气温是多少？"}]

# pass functions directly as tools in the tools list or as a JSON schema
response = ollama.chat(model=glm_model, messages=messages, tools=all_tools.values(), think=True)
messages.append(response.message)
for iteration in range(max_iterations):
  if tool_calls := response.message.tool_calls:
    # only recommended for models which only return a single tool call
    for tool_call in tool_calls:
      # add the tool result to the messages
      function_name = tool_call.function.name
      function_args = tool_call.function.arguments
      if function_name == 'get_current_time' and len(function_args)==0:
          function_args={'timezone':'Asia/Shanghai'}
      # Call the function and get the result
      tool_result = all_tools.get(function_name, None)(**function_args)
      
      # Add the tool result to messages
      messages.append({"role": "tool", "name": function_name, "content": tool_result})
      response = ollama.chat(model=glm_model, messages=messages, tools=all_tools.values(), think=True)
      messages.append(response.message)
  else:
    break
final_response = ollama.chat(model=glm_model, messages=messages, tools=all_tools.values(), think=True)
print_html(final_response.message.content)

In [10]:
messages

[{'role': 'user', 'content': '现在几点了？气温是多少？'},
 Message(role='assistant', content='我来帮您查看当前时间和气温。', thinking='用户问现在几点了以及气温是多少。要回答这两个问题，我需要：\n\n1. 获取当前时间 - 使用 get_current_time 函数\n2. 获取气温 - 使用 get_weather_from_ip 函数\n\n这两个函数都不需要必需的参数，可以直接调用。', images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_current_time', arguments={'timezone': 'Asia/Shanghai'})), ToolCall(function=Function(name='get_weather_from_ip', arguments={}))]),
 {'role': 'tool', 'name': 'get_current_time', 'content': '10:49:16'},
 Message(role='assistant', content='根据查询结果：\n\n**当前时间：** 10:49:16（北京时间）\n\n**气温信息：** 很抱歉，我无法获取到当前的气温数据。可能是由于位置信息获取问题或服务暂时不可用。\n\n建议您可以通过其他方式（如天气预报应用或网站）查询当前气温。如果您知道具体的时区，我也可以帮您查询该时区的当前时间。', thinking='从结果来看：\n1. 当前时间是 10:49:16\n2. 但是 get_weather_from_ip 的结果似乎没有返回天气信息，只返回了时间。我需要向用户说明情况。\n\n让我检查一下 weather 函数的返回值。看起来 weather 函数可能没有正常工作或者返回的数据格式不符合预期。', images=None, tool_name=None, tool_calls=None),
 {'role': 'tool',
  'name': 'get_weather_from_ip',
  'content': 'Current: 9.1°C, 

### openai-api

In [30]:
from openai import OpenAI

In [31]:
client = OpenAI(base_url="http://127.0.0.1:8080",api_key="sk-no-key-required")

In [32]:
qwen35_model = 'qwen3.5-35b-a3b-q4km'

In [33]:
completion = client.chat.completions.create(
    model=qwen35_model,
    messages=[
        {"role": "system", "content": "你是一个有用的助手，可以帮助用户完成各种任务。"},
        {"role": "user", "content": "请帮我计算 15 * 23 + 7"}
    ]
)

In [34]:
print(completion.choices[0].message.content)

计算过程如下：

1.  先算乘法：$15 \times 23 = 345$
2.  再算加法：$345 + 7 = 352$

所以，$15 \times 23 + 7 = 352$。


In [35]:
15 * 23 + 7

352

In [36]:
# 定义 OpenAI tools 格式
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取指定时区的当前时间，默认时区为 Asia/Shanghai",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "时区名称，例如: Asia/Shanghai, America/New_York, Europe/London"
                    }
                },
                "required": ["timezone"]    
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_from_ip",
            "description": "根据 IP 地址获取天气信息（模拟数据）",
            "parameters": {
                "type": "object",
                "properties": {
                    "ip": {
                        "type": "string",
                        "description": "IP 地址，可选参数，不提供时返回默认位置的天气"
                    }
                },
                "required": []
            }
        }
    }
]

In [37]:
# 定义一个通用的工具处理函数
def handle_tool_calls_with_multiple_tools(response, messages, client, tools, max_iterations=5):
    """处理工具调用的循环，支持多个工具"""
    for iteration in range(max_iterations):
        if response.choices[0].message.tool_calls:
            tool_call = response.choices[0].message.tool_calls[0]
            tool_name = tool_call.function.name
            
            # 解析参数
            if tool_call.function.arguments:
                args = json.loads(tool_call.function.arguments)
            else:
                args = {}
            
            # 执行对应的工具函数
            if tool_name == "get_current_time":
                result = get_current_time(args.get("timezone","Asia/Shanghai"))
            elif tool_name == "get_weather_from_ip":
                result = get_weather_from_ip()
            elif tool_name == "write_txt_file":
                result = write_txt_file(args.get("file_path"), args.get("content"))
            elif tool_name == "generate_qr_code":
                result = generate_qr_code(args.get("data"), args.get("filename"), args.get("image_path"))
            else:
                result = f"未知工具: {tool_name}"
            
            print(f"工具 {tool_name} 执行结果: {result}")
            
            # 将工具结果返回给LLM
            messages.append({
                "role": "assistant",
                "content": "",
                "tool_calls": [tool_call.model_dump()]
            })
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })
            
            # 获取下一轮响应
            response = client.chat.completions.create(
                model=qwen35_model,
                messages=messages,
                tools=tools
            )
        else:
            break
    return response

# 测试天气查询
prompt = "请问现在是几点了？可以帮我查询我所在位置的天气吗？"

messages = [
    {"role": "user", "content": prompt}
]

response = client.chat.completions.create(
    model=qwen35_model,
    messages=messages,
    tools=tools
)

# 处理工具调用
final_response = handle_tool_calls_with_multiple_tools(response, messages, client, tools, max_iterations=5)

print_html(f"\n最终响应: {final_response.choices[0].message.content}")

工具 get_current_time 执行结果: 21:40:27
工具 get_weather_from_ip 执行结果: Current: 17.2°C, High: 22.4°C, Low: 17.2°C
